In [ ]:
import os
import json
import random
import shutil
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm

In [ ]:
MANUFACTURER_MAP = [
	("Rocky Mountain Construction", "RMC"),
	("Bolliger & Mabillard", "BM"),
	("Vekoma", "Vekoma"),
	("Schwarzkopf", "Schwarzkopf"),
	("Gerstlauer Amusement Rides GmbH", "Gerstlauer"),
	("Gerstlauer", "Gerstlauer")
]

COASTERDB_FILE = "../../rcdb-api/db/coasters.json"

TARGET_YEAR = 2015
MAX_IMAGES_PER_MAKE = 300  # Cap per manufacturer to maintain a balanced dataset
TRAIN_SPLIT_RATIO = 0.8    # 80% Train, 20% Blind Test

DATASET_DIR = "dataset"
for split in ["train", "test"]:
	for _, folder_name in MANUFACTURER_MAP:
		os.makedirs(os.path.join(DATASET_DIR, split, folder_name), exist_ok=True)

In [ ]:
existing_samples = set()

for split in ["train", "test"]:
	for _, folder in MANUFACTURER_MAP:
		folder_path = os.path.join(DATASET_DIR, split, folder)
		if os.path.exists(folder_path):
			existing_samples.update(os.listdir(folder_path))

with open(COASTERDB_FILE, "r", encoding="utf-8") as f:
	coasters_data = json.load(f)

scouting_vault = {folder: [] for _, folder in MANUFACTURER_MAP}

for coaster in coasters_data:
	make = coaster.get("make", "").lower()

	target_folder = next((folder for actual, folder in MANUFACTURER_MAP if actual.lower() in make), None)
	if not target_folder:
		continue

	all_pics = [coaster["mainPicture"]] if coaster.get("mainPicture") else []
	all_pics.extend(coaster.get("pictures", []))

	for pic in all_pics:
		raw_url = pic.get("url", "")
		copy_date = pic.get("copyDate", "")

		if not copy_date or not copy_date[:4].isdigit() or int(copy_date[:4]) < TARGET_YEAR:
			continue

		if "undefined/" in raw_url:
			filename = f"c{coaster['id']}_i{pic.get('id')}.jpg"
			if filename in existing_samples:
				continue

			resolved_url = raw_url.replace("undefined/", "https://rcdb.com/")
			scouting_vault[target_folder].append((filename, resolved_url))

download_queue = {folder: [] for _, folder in MANUFACTURER_MAP}

TARGET_TRAIN_COUNT = int(MAX_IMAGES_PER_MAKE * TRAIN_SPLIT_RATIO)
TARGET_TEST_COUNT = MAX_IMAGES_PER_MAKE - TARGET_TRAIN_COUNT

for folder, candidates in scouting_vault.items():
	train_path = os.path.join(DATASET_DIR, "train", folder)
	test_path = os.path.join(DATASET_DIR, "test", folder)
	
	current_train = len(os.listdir(train_path)) if os.path.exists(train_path) else 0
	current_test = len(os.listdir(test_path)) if os.path.exists(test_path) else 0
	
	needed_train = max(0, TARGET_TRAIN_COUNT - current_train)
	needed_test = max(0, TARGET_TEST_COUNT - current_test)

	random.shuffle(candidates)
	
	allocated_train = candidates[:needed_train]
	allocated_test = candidates[needed_train : needed_train + needed_test]

	for file_name, url in allocated_train:
		download_queue[folder].append((file_name, url, "train"))
	for file_name, url in allocated_test:
		download_queue[folder].append((file_name, url, "test"))
		
	print(f"\t{folder:<15}: Train Images Needed: {needed_train:<4} | Test Images Needed: {needed_test:<4}")

In [ ]:
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) CoasterVision/1.0"}

all_tasks = sum(len(tasks) for tasks in download_queue.values())

if all_tasks == 0:
	print("No new images need to be downloaded.")
else:
	summary_counts = {folder: {"train": 0, "test": 0} for folder in download_queue.keys()}
	
	queue_generator = (
		(folder, filename, url, split_name)
		for folder, tasks in download_queue.items()
		for filename, url, split_name in tasks
	)
	
	for folder, filename, url, split_name in tqdm(queue_generator, total=all_tasks, desc="Downloading Assets", unit="img"):
		destination_path = os.path.join(DATASET_DIR, split_name, folder, filename)
		os.makedirs(os.path.dirname(destination_path), exist_ok=True)
		
		try:
			response = requests.get(url, headers=headers, timeout=10)
			if response.status_code != 200:
				tqdm.write(f"[HTTP {response.status_code}] {filename}")
				continue
				
			content_type = response.headers.get("Content-Type", "")
			if "image" not in content_type:
				tqdm.write(f"[INVALID TYPE] {filename} ({content_type})")
				continue
				
			img = Image.open(BytesIO(response.content))
			if img.mode != 'RGB':
				img = img.convert('RGB')
				
			img = img.resize((224, 224), Image.Resampling.BILINEAR)

			img.save(destination_path, "JPEG", quality=95)
			
			summary_counts[folder][split_name] += 1
			
		except Exception as e:
			tqdm.write(f"[FAILURE] {filename}: {type(e).__name__}")

	print("\nDownload Summary:")
	for folder, counts in summary_counts.items():
		if counts["train"] > 0 or counts["test"] > 0:
			print(f"\t{folder:<15}: Train Images Added: {counts['train']:<4} | Test Images Added: {counts['test']:<4}")